# Model Governance and Audit Trail with MLflow

## Topic Goal

This notebook properly implements the expected **Model Governance and Audit Trail** use case with MLflow.

In enterprise MLOps, a model is not ready for production only because it has good metrics.  
It also needs governance metadata, audit trail, approval information, ownership details, and traceability.

This notebook demonstrates:

1. Load customer churn dataset.
2. Train and evaluate a model.
3. Define governance metadata:
   - model owner
   - business unit
   - model purpose
   - risk level
   - data sensitivity
   - approval status
   - reviewer
   - approval date
4. Create governance checklist.
5. Create audit metadata JSON.
6. Create model card style artifact.
7. Log governance metadata as MLflow params and tags.
8. Log audit artifacts to MLflow.
9. Log model with input example and signature in the **same MLflow run**.
10. Create `model_uri` from the same active run.
11. Register the model using the same-run `model_uri`.
12. Assign an MLflow Registry alias based on governance status.

This is a production-style pattern for auditability, compliance, and model lifecycle management.


## 1. Import Libraries

We import MLflow, pandas, scikit-learn, JSON utilities, and `MlflowClient`.

`MlflowClient` is used to assign a registry alias such as:

```text
pending_review
approved_candidate
rejected
```


In [ ]:
# Import os to create folders and manage local paths.
import os

# Import json to save governance and audit metadata.
import json

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import datetime to timestamp governance artifacts.
from datetime import datetime

# Import MLflow for tracking, logging, and registry.
import mlflow

# Import MLflow sklearn flavor for logging sklearn pipelines.
import mlflow.sklearn

# Import MlflowClient for registry alias operations.
from mlflow.tracking import MlflowClient

# Import infer_signature to capture input/output model schema.
from mlflow.models.signature import infer_signature

# Import pandas for data processing.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import train_test_split for splitting data.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for preprocessing numerical and categorical columns.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical columns.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical columns.
from sklearn.preprocessing import StandardScaler

# Import Pipeline to combine preprocessing and model.
from sklearn.pipeline import Pipeline

# Import RandomForestClassifier for churn prediction.
from sklearn.ensemble import RandomForestClassifier

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix


## 2. Configure MLflow and Local Folders

This notebook uses local file-based MLflow tracking.

All governance and audit artifacts are created under:

```text
outputs/
```


In [ ]:
# Define experiment name.
EXPERIMENT_NAME = "advanced_18_model_governance_audit"

# Define run name.
RUN_NAME = "18_model_governance_audit_same_run_usecase"

# Remove inherited MLflow Project experiment ID if present.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited MLflow run ID if present.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create outputs folder for governance and audit artifacts.
os.makedirs("outputs", exist_ok=True)

# Create artifacts folder for support files.
os.makedirs("artifacts", exist_ok=True)

# Configure MLflow local tracking.
mlflow.set_tracking_uri("file:./mlruns")

# Define dataset path.
DATA_PATH = "datasets/customer_churn_500.csv"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Create MLflow client for registry alias operations.
client = MlflowClient()

# Print setup details.
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", RUN_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)


## 3. Load Dataset

We use the customer churn dataset.

The target column is:

```text
churn
```


In [ ]:
# Check whether dataset exists.
if not os.path.exists(DATA_PATH):
    # Raise clear error if dataset is missing.
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# Load dataset.
df = pd.read_csv(DATA_PATH)

# Display first five rows.
display(df.head())

# Define target column.
target_column = "churn"

# Define ID column.
id_column = "customer_id"

# Print dataset shape.
print("Dataset shape:", df.shape)


## 4. Define Governance Metadata

This section defines governance details that enterprise teams usually require.

These values are logged as MLflow params, tags, and artifacts.


In [ ]:
# Define model owner.
model_owner = "data_science_team"

# Define business unit.
business_unit = "customer_retention"

# Define model purpose.
model_purpose = "Predict customer churn risk for retention campaign prioritization"

# Define use case category.
use_case_category = "customer_analytics"

# Define risk level.
risk_level = "medium"

# Define data sensitivity.
data_sensitivity = "internal_customer_behavior_data"

# Define compliance scope.
compliance_scope = "internal_governance_review"

# Define approval status.
approval_status = "pending_review"

# Define reviewer name or group.
reviewer = "ml_governance_committee"

# Define approval date.
approval_date = "not_approved_yet"

# Define model lifecycle stage.
lifecycle_stage = "candidate"

# Define monitoring requirement.
monitoring_required = True

# Define retraining policy.
retraining_policy = "retrain_or_review_if_f1_drops_by_5_percent_or_data_drift_detected"

# Print governance metadata.
print("Model owner:", model_owner)
print("Business unit:", business_unit)
print("Risk level:", risk_level)
print("Approval status:", approval_status)
print("Reviewer:", reviewer)


## 5. Create Governance Checklist

A governance checklist helps ensure that required controls are reviewed before production.

For this demo, all mandatory checks must pass before the model can receive an `approved_candidate` alias.


In [ ]:
# Define governance checklist.
governance_checklist = {
    "dataset_available": os.path.exists(DATA_PATH),
    "target_column_present": target_column in df.columns,
    "no_empty_dataset": len(df) > 0,
    "owner_defined": model_owner is not None and model_owner != "",
    "business_unit_defined": business_unit is not None and business_unit != "",
    "risk_level_defined": risk_level in ["low", "medium", "high"],
    "reviewer_defined": reviewer is not None and reviewer != "",
    "monitoring_required": monitoring_required,
    "approval_status_valid": approval_status in ["pending_review", "approved", "rejected"],
}

# Check whether all governance checks passed.
governance_checks_passed = all(governance_checklist.values())

# Print checklist.
for check_name, check_status in governance_checklist.items():
    print(check_name, "->", check_status)

# Print overall governance checklist status.
print("Governance checks passed:", governance_checks_passed)


## 6. Prepare Features, Target, and Preprocessing

We prepare a standard customer churn classification pipeline.

The governance metadata will be logged together with the model run.


In [ ]:
# Create feature matrix by removing ID and target columns.
X = df.drop(columns=[id_column, target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Scale numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split dataset into train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print feature details.
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 7. Train and Evaluate Model

We train a Random Forest churn model and calculate standard classification metrics.

These metrics become part of the model audit trail.


In [ ]:
# Create Random Forest classifier.
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    random_state=42
)

# Create complete pipeline with preprocessing and model.
pipeline = Pipeline(steps=[
    # Apply preprocessing to raw input data.
    ("preprocessor", preprocessor),

    # Train Random Forest model.
    ("model", model),
])

# Train pipeline.
pipeline.fit(X_train, y_train)

# Generate predictions.
predictions = pipeline.predict(X_test)

# Calculate accuracy.
accuracy = accuracy_score(y_test, predictions)

# Calculate precision.
precision = precision_score(y_test, predictions, zero_division=0)

# Calculate recall.
recall = recall_score(y_test, predictions, zero_division=0)

# Calculate F1-score.
f1 = f1_score(y_test, predictions, zero_division=0)

# Create classification report.
report = classification_report(y_test, predictions, zero_division=0)

# Create confusion matrix.
cm = confusion_matrix(y_test, predictions)

# Print metrics.
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print(report)


## 8. Create Audit Artifacts

Governance and audit artifacts are saved as files and later logged to MLflow.

Artifacts created:

- governance checklist JSON
- audit metadata JSON
- model card markdown
- classification report
- confusion matrix CSV


In [ ]:
# Create audit timestamp.
audit_timestamp = datetime.now().isoformat()

# Create audit metadata dictionary.
audit_metadata = {
    "audit_timestamp": audit_timestamp,
    "experiment_name": EXPERIMENT_NAME,
    "model_owner": model_owner,
    "business_unit": business_unit,
    "model_purpose": model_purpose,
    "use_case_category": use_case_category,
    "risk_level": risk_level,
    "data_sensitivity": data_sensitivity,
    "compliance_scope": compliance_scope,
    "approval_status": approval_status,
    "reviewer": reviewer,
    "approval_date": approval_date,
    "lifecycle_stage": lifecycle_stage,
    "monitoring_required": monitoring_required,
    "retraining_policy": retraining_policy,
    "dataset_path": DATA_PATH,
    "target_column": target_column,
    "feature_count": X.shape[1],
    "categorical_columns": categorical_columns,
    "numerical_columns": numerical_columns,
    "metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
    },
    "governance_checklist": governance_checklist,
    "governance_checks_passed": governance_checks_passed,
}

# Save governance checklist.
with open("outputs/governance_checklist.json", "w") as file:
    json.dump(governance_checklist, file, indent=4)

# Save audit metadata.
with open("outputs/audit_metadata.json", "w") as file:
    json.dump(audit_metadata, file, indent=4)

# Save classification report.
with open("outputs/classification_report.txt", "w") as file:
    file.write(report)

# Save confusion matrix CSV.
confusion_matrix_df = pd.DataFrame(
    cm,
    index=["actual_0", "actual_1"],
    columns=["predicted_0", "predicted_1"]
)
confusion_matrix_df.to_csv("outputs/confusion_matrix.csv")

# Create model card content.
model_card = f'''# Model Card: Customer Churn Model

## Model Overview
- Model name: {REGISTERED_MODEL_NAME}
- Model owner: {model_owner}
- Business unit: {business_unit}
- Use case: {model_purpose}
- Lifecycle stage: {lifecycle_stage}

## Governance
- Risk level: {risk_level}
- Data sensitivity: {data_sensitivity}
- Compliance scope: {compliance_scope}
- Approval status: {approval_status}
- Reviewer: {reviewer}
- Approval date: {approval_date}

## Performance
- Accuracy: {accuracy}
- Precision: {precision}
- Recall: {recall}
- F1-score: {f1}

## Monitoring
- Monitoring required: {monitoring_required}
- Retraining policy: {retraining_policy}

## Audit
- Audit timestamp: {audit_timestamp}
- Governance checks passed: {governance_checks_passed}
'''

# Save model card.
with open("outputs/model_card.md", "w") as file:
    file.write(model_card)

# Print generated artifacts.
print("Generated audit artifacts:")
print("- outputs/governance_checklist.json")
print("- outputs/audit_metadata.json")
print("- outputs/model_card.md")
print("- outputs/classification_report.txt")
print("- outputs/confusion_matrix.csv")


## 9. Determine Registration Governance Alias

The model will still be registered for lifecycle tracking, but the registry alias depends on governance status.

For example:

```text
approved_candidate
pending_review
rejected
```

In a real enterprise workflow, only approved models would move to production.


In [ ]:
# Determine registry alias based on governance status.
if approval_status == "approved" and governance_checks_passed:
    # Alias for approved model candidate.
    registry_alias = "approved_candidate"
elif approval_status == "rejected":
    # Alias for rejected model.
    registry_alias = "rejected"
else:
    # Default alias for model waiting for review.
    registry_alias = "pending_review"

# Print alias decision.
print("Registry alias to assign:", registry_alias)


## 10. Infer Model Signature

Model signature captures the input and output contract.

This supports auditability because future users can understand the expected input schema.


In [ ]:
# Create input example from test data.
input_example = X_test.head(5)

# Generate sample predictions for signature.
sample_predictions = pipeline.predict(input_example)

# Infer model signature.
signature = infer_signature(input_example, sample_predictions)

# Display input example.
display(input_example)

# Print sample predictions.
print("Sample predictions:", sample_predictions)


## 11. Same-Run MLflow Logging with Governance and Audit Trail

This is the main production-style governance run.

Inside the **same MLflow run**, we log:

- model metrics
- governance metadata as params and tags
- audit artifacts
- model card
- model with signature
- input example
- model URI

Then we register the model using the same `model_uri`.


In [ ]:
# Start the SAME MLflow run for governance, audit, model, signature, and model URI.
with mlflow.start_run(run_name=RUN_NAME):

    # -----------------------------
    # Core model metadata
    # -----------------------------

    # Log dataset path.
    mlflow.log_param("dataset", DATA_PATH)

    # Log topic.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # -----------------------------
    # Governance parameters
    # -----------------------------

    # Log model owner.
    mlflow.log_param("model_owner", model_owner)

    # Log business unit.
    mlflow.log_param("business_unit", business_unit)

    # Log use case category.
    mlflow.log_param("use_case_category", use_case_category)

    # Log risk level.
    mlflow.log_param("risk_level", risk_level)

    # Log data sensitivity.
    mlflow.log_param("data_sensitivity", data_sensitivity)

    # Log compliance scope.
    mlflow.log_param("compliance_scope", compliance_scope)

    # Log approval status.
    mlflow.log_param("approval_status", approval_status)

    # Log reviewer.
    mlflow.log_param("reviewer", reviewer)

    # Log lifecycle stage.
    mlflow.log_param("lifecycle_stage", lifecycle_stage)

    # Log whether monitoring is required.
    mlflow.log_param("monitoring_required", monitoring_required)

    # Log governance checklist result.
    mlflow.log_param("governance_checks_passed", governance_checks_passed)

    # Log registry alias decision.
    mlflow.log_param("registry_alias", registry_alias)

    # -----------------------------
    # Metrics
    # -----------------------------

    # Log accuracy.
    mlflow.log_metric("accuracy", accuracy)

    # Log precision.
    mlflow.log_metric("precision", precision)

    # Log recall.
    mlflow.log_metric("recall", recall)

    # Log F1-score.
    mlflow.log_metric("f1_score", f1)

    # -----------------------------
    # Governance tags
    # -----------------------------

    # Set owner tag.
    mlflow.set_tag("model_owner", model_owner)

    # Set business unit tag.
    mlflow.set_tag("business_unit", business_unit)

    # Set risk level tag.
    mlflow.set_tag("risk_level", risk_level)

    # Set approval status tag.
    mlflow.set_tag("approval_status", approval_status)

    # Set reviewer tag.
    mlflow.set_tag("reviewer", reviewer)

    # Set lifecycle stage tag.
    mlflow.set_tag("lifecycle_stage", lifecycle_stage)

    # Set data sensitivity tag.
    mlflow.set_tag("data_sensitivity", data_sensitivity)

    # Set run purpose tag.
    mlflow.set_tag("run_purpose", "model_governance_and_audit_trail")

    # Set governance result tag.
    mlflow.set_tag("governance_checks_passed", str(governance_checks_passed))

    # -----------------------------
    # Audit artifacts
    # -----------------------------

    # Log governance checklist.
    mlflow.log_artifact("outputs/governance_checklist.json")

    # Log audit metadata.
    mlflow.log_artifact("outputs/audit_metadata.json")

    # Log model card.
    mlflow.log_artifact("outputs/model_card.md")

    # Log classification report.
    mlflow.log_artifact("outputs/classification_report.txt")

    # Log confusion matrix.
    mlflow.log_artifact("outputs/confusion_matrix.csv")

    # -----------------------------
    # Model artifact
    # -----------------------------

    # Log model with input example and signature in the SAME run.
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Register model using the same-run model URI.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

# Print model registration details.
print("Same-run model URI:", model_uri)
print("Registered model name:", registered_model.name)
print("Registered model version:", registered_model.version)


## 12. Assign Registry Alias Based on Governance Status

After registration, we assign a registry alias based on governance state.

This helps users access models by governance status instead of hardcoding version numbers.


In [ ]:
# Assign registry alias to the registered model version.
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias=registry_alias,
    version=registered_model.version
)

# Print alias assignment details.
print("Assigned alias:", registry_alias)
print("Model name:", REGISTERED_MODEL_NAME)
print("Model version:", registered_model.version)

# Print alias URI.
print("Alias URI:", f"models:/{REGISTERED_MODEL_NAME}@{registry_alias}")


## 13. Verify Governance Artifacts

This section verifies that all audit artifacts exist locally.


In [ ]:
# Define expected governance artifacts.
expected_artifacts = [
    "outputs/governance_checklist.json",
    "outputs/audit_metadata.json",
    "outputs/model_card.md",
    "outputs/classification_report.txt",
    "outputs/confusion_matrix.csv",
]

# Verify each artifact.
for artifact_path in expected_artifacts:
    print(artifact_path, "->", os.path.exists(artifact_path))



- Governance is about accountability, ownership, and control.
- Audit trail explains who created the model, why it was created, and how it was evaluated.
- MLflow params are useful for searchable metadata.
- MLflow tags are useful for governance and lifecycle filtering.
- Artifacts store detailed audit evidence such as model card, checklist, and reports.
- Registry aliases can represent governance status such as `pending_review` or `approved_candidate`.
- A model can be registered for tracking but should not be promoted to production unless governance approval is complete.
- This notebook keeps model metrics, audit metadata, signature, and registered model lineage together.
